# MUS-Net: Multi-Window Uncertainty-Guided Network (L4 24GB GPU)
### Paper: *Multi-window uncertainty-guided network for lung nodule CT segmentation* (Lin et al., AEJ 2025)

**LIDC-IDRI Solution for Colab Disk Constraints**:
- Raw DICOMs are ~125GB (exceeding Colab's ~100GB limit).
- **Streaming Solution (Cách 2)**: Downloads 1 patient CT scan at a time, crops 128x128 patches in `int16` HU, saves directly to Google Drive, and **deletes the raw DICOM immediately**!
- Total processed dataset is only **~150MB** (fits in free 15GB Google Drive effortlessly).
- Colab disk usage never exceeds ~500MB!

## 1. Mount Google Drive & Verify Hardware

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

# Check GPU (NVIDIA L4 24GB)
!nvidia-smi

## 2. Clone Repository & Install Dependencies

In [ ]:
# Option A: Clone from your GitHub repo
# !git clone <YOUR_GITHUB_REPO_URL> /content/mus_net
# %cd /content/mus_net

# Option B: If unzipping uploaded archive
!unzip -q /content/mus_net.zip -d /content/mus_net
%cd /content/mus_net

!pip install -q -r requirements.txt
!pip install -q requests

## 3. LIDC-IDRI Streaming Preprocessor (Cách 2 - Khuyên dùng)
Tải trực tiếp từng ca bệnh nhân qua TCIA REST API -> Cắt patch 128x128 `int16` HU (chỉ lấy Annotation 1) -> Lưu vào Google Drive -> Xóa ngay DICOM gốc!
- Nếu phiên làm việc bị gián đoạn, script tự động bỏ qua những bệnh nhân đã xử lý xong.

In [ ]:
DRIVE_DATA = '/content/drive/MyDrive/lidc_preprocessed'

# Chạy streaming download: tải tới đâu xử lý và lưu vào Drive tới đó, xóa DICOM ngay
# Bỏ --max_patients hoặc đặt số lượng tùy ý (ví dụ: 100 hoặc chạy hết 1018 ca)
!python -m mus_net.data.run_stream_preprocessor \
  --output_dir {DRIVE_DATA} \
  --temp_dir /content/temp_dicom \
  --patch_size 128

## 4. Copy Dữ liệu từ Google Drive sang Colab Local NVMe SSD (~5 giây)
Đưa dataset (~150MB) vào ổ SSD `/content/data/lidc_preprocessed` để DataLoader nạp vào GPU L4 ở tốc độ tối đa.

In [ ]:
LOCAL_DATA = '/content/data/lidc_preprocessed'
!mkdir -p {LOCAL_DATA}
!cp -r {DRIVE_DATA}/* {LOCAL_DATA}/

# Kiểm tra số lượng lát cắt đã sẵn sàng
!python -m mus_net.data.download_lidc --output_dir {LOCAL_DATA}

## 5. Chạy PyTest Kiểm tra Module & Shape
Xác nhận các nhánh FEM, UIM, FFM (Sobel, Otsu STE, Gabor), MWM và MUS-Net nhận [B,1,128,128] và xuất [B,1,128,128].

In [ ]:
!pytest tests/ -v

## 6. Huấn luyện MUS-Net (Lưu Real-time vào Google Drive & Auto-Resume)
Tất cả checkpoint (`best_model_fold_0.pth`, `latest_model_fold_0.pth`) và log CSV được ghi trực tiếp vào Drive mỗi epoch.

In [ ]:
DRIVE_RUNS = '/content/drive/MyDrive/mus_net_experiments'

# Huấn luyện Fold 0 (hoặc bỏ --fold để chạy toàn bộ 5-Fold Cross Validation)
!python train.py \
  --config configs/default.yaml \
  --fold 0 \
  --drive_dir {DRIVE_RUNS}

## 7. Đánh giá Định lượng & Xuất Ảnh Multi-Window Mask (MWM) và Grad-CAM

In [ ]:
!python evaluate.py \
  --config configs/default.yaml \
  --drive_dir {DRIVE_RUNS}

## 8. Hiển thị Trực quan hóa MWM và Grad-CAM trong Notebook

In [ ]:
import os
from IPython.display import Image, display

vis_path = f'{DRIVE_RUNS}/visualizations/sample_eval.png'
if os.path.exists(vis_path):
    display(Image(vis_path))
else:
    print('Bản đồ trực quan hóa sẽ xuất hiện tại đây sau khi lệnh evaluate.py hoàn tất.')